In [ ]:
#Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import sys

#Root
project_root = Path.cwd().parent
sys.path.append(str(project_root))

#Utils
from src.utils.db import get_engine
from src.utils.logger import setup_logger

logger = setup_logger(__name__)
engine = get_engine()

logger.info("Environment ready")

In [ ]:
#Stores sales
query = """ 
    SELECT 
        stores.store_nbr,
        stores.city,
        stores.state,
        stores.type,
        stores.cluster,
        COUNT(*) as number_transactions, 
        SUM(train.unit_sales) as total_sales,
        COUNT(DISTINCT train.item_nbr) as unique_items_sold,
        COUNT(DISTINCT DATE(train.date)) as days_active

    FROM train
    JOIN stores
    ON train.store_nbr = stores.store_nbr
    GROUP BY stores.store_nbr,stores.city,stores.state,stores.cluster
    ORDER BY total_sales DESC;
"""

logger.info("Loading store sales data...")
stores_performance = pd.read_sql(query,engine)
logger.info(f"Loaded sale from {len(stores_performance)} stores")

print("Stores Performance:\n")
stores_performance.head()

In [ ]:
#Stores by top and bottom sales
top_10_stores = stores_performance.head(10)
bottom_10_stores = stores_performance.tail(10)

stores = pd.read_sql("Select * FROM stores",engine)

n_stores = len(stores)

print(f"\n📊 Product Concentration:")
total_sales_all = stores_performance['total_sales'].sum()
top_5_sales = stores_performance.head(5)['total_sales'].sum()
top_10_sales = stores_performance.head(10)['total_sales'].sum()
print(f"\nTotal stores: {n_stores}")
print(f"Top 5 stores({5/n_stores*100:.1f}% total) stores: {top_5_sales/total_sales_all*100:.1f}% of total sales")
print(f"Top 10 stores({10/n_stores*100:.1f}% total) stores: {top_10_sales/total_sales_all*100:.1f}% of total sales")


fig = make_subplots( rows = 1, cols= 2, subplot_titles=('Top 10 stores',"Bottom 10 stores"))

fig.add_trace(
    go.Bar(x=top_10_stores['store_nbr'].astype(str), y=top_10_stores['total_sales'],marker_color = "green", text=top_10_stores['total_sales'],name='Top 10'),
    row=1,col=1
)

fig.add_trace(
    go.Bar(x=bottom_10_stores['store_nbr'].astype(str), y=bottom_10_stores['total_sales'],marker_color = "red",text=bottom_10_stores['total_sales'],name = 'Bottom 10'),
    row=1,col=2
)

fig.update_layout(separators=",.",
                  xaxis_title='Store Number',yaxis_title='Total Sales',
                  xaxis2_title='Store Number',yaxis2_title='Total Sales',
                  )
fig.update_traces(texttemplate='%{text:,.0f}')
fig.show()

In [ ]:
# Type of stores with most sales

type_store_perfomance = stores_performance.groupby('type').agg({
    "store_nbr":'count',
    'total_sales':'sum'
})
type_store_perfomance.columns = ['Num_Stores', 'Total_Sales']

type_store_perfomance['sale_per_store'] = (type_store_perfomance['Total_Sales']/type_store_perfomance['Num_Stores']).round()

type_store_perfomance = type_store_perfomance.sort_values('sale_per_store', ascending=False)

fig = px.bar(
    type_store_perfomance.reset_index(),'type','sale_per_store',title="Average sales per Store by Type",text='sale_per_store')

fig.update_layout(separators=",.", xaxis_title='Type',yaxis_title='Avg Sales')
fig.update_traces(texttemplate='%{text:,.0f}')

fig.show()

type_store_perfomance.head()

In [ ]:
# Cities with highest-selling stores
cities_store_performance = stores_performance.groupby('city').agg({
    'store_nbr':'count',
    'total_sales':'sum',
    'number_transactions':'sum'
}).round(0)

cities_store_performance.columns = ['Num_Stores', 'Total_Sales', 'Total_Transactions']

cities_store_performance = cities_store_performance.sort_values('Total_Sales',ascending=False).head(10)

fig = px.bar(
    cities_store_performance.reset_index(),'city','Total_Sales',title='Top 10 Cities by Total Sales', text='Total_Sales')

fig.update_layout(separators=",.", xaxis_title='City',yaxis_title='Total Sales',)
fig.update_traces(texttemplate='%{text:,.0f}')

fig.show()

In [ ]:
#Family items

items = pd.read_sql("SELECT * FROM items", engine)
query = """ 
    SELECT
        items.family,
        COUNT(*) as num_transactions,
        ROUND(SUM(train.unit_sales),0) as total_sales,
        ROUND(AVG(train.unit_sales),2) as avg_sales,
        COUNT(DISTINCT train.item_nbr) as unique_items,
        COUNT(DISTINCT train.store_nbr) as stores_selling
    FROM train
    JOIN items 
    ON train.item_nbr = items.item_nbr
    GROUP BY items.family
    ORDER BY total_sales DESC;
"""
logger.info("Loading items data...")
family_performace = pd.read_sql(query,engine)
logger.info(f"Loaded {len(family_performace)} family of items")

print("\n Product Family Performance:")
display(family_performace.head(15))


In [ ]:
top_15_families = family_performace.head(15)

fig = px.bar(top_15_families, x='family',y='total_sales', title='Top 15 Product Families by Total Sales', text='total_sales')
fig.update_layout(separators=",.", xaxis_title='Item Family',yaxis_title='Total Sales')
fig.update_traces(texttemplate='%{text:,.0f}')

fig.show()

print(f"\n📊 Product Concentration:")
total_sales_all = family_performace['total_sales'].sum()
top_5_sales = family_performace.head(5)['total_sales'].sum()
top_10_sales = family_performace.head(10)['total_sales'].sum()

print(f"Top 5 families: {top_5_sales/total_sales_all*100:.1f}% of total sales")
print(f"Top 10 families: {top_10_sales/total_sales_all*100:.1f}% of total sales")


In [ ]:
#Perishable items

query = """
    SELECT 
        items.perishable::INTEGER as is_perishable,
        COUNT(*) as num_transactions,
        ROUND(SUM(train.unit_sales),0) as total_sales,
        COUNT(DISTINCT items.item_nbr) as num_items
    FROM train
    JOIN items
    ON train.item_nbr = items.item_nbr
    GROUP BY 1
    ORDER BY 1
"""
logger.info("Loading perishable items data...")
perishable_items = pd.read_sql(query,engine)
logger.info("Loaded Perishable items data")


In [ ]:
# Perishable items vs Non-Perishables
perishable_items['is_perishable'] = perishable_items['is_perishable'].map({
    0: 'Non-Perishable',
    1: 'Perishable'
})
print("\nPerishable vs Non-Perishable items:")
display(perishable_items.head())

fig = make_subplots(
    rows=1, cols=2, specs=[[{"type": "pie"}, {"type": "bar"}]], subplot_titles=('Sales Distribution', 'Number of Transactions')
)

fig.add_trace(
    go.Pie(
        labels=perishable_items['is_perishable'],
        values=perishable_items['total_sales'],
        marker_colors=['#ff7f0e', '#1f77b4']
    ), 
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=perishable_items['is_perishable'], y=perishable_items['num_transactions'], text=perishable_items['num_transactions'],marker_color=['#ff7f0e', '#1f77b4']),
    row=1, col=2
)

fig.update_layout(separators=",.", showlegend=False, xaxis2_title='Perishability',yaxis2_title='Total Transactions',)
fig.update_traces(texttemplate='%{text:,.0f}', selector=dict(type='bar'))


fig.show()


In [ ]:
#Promo impact

query = """ 
    SELECT 
        COALESCE(onpromotion, false) as on_promotion,
        ROUND(SUM(unit_sales), 0) AS total_sales,
        ROUND(AVG(unit_sales), 2) AS avg_sales_per_transaction,
        COUNT(*) AS num_transactions
    FROM train
    GROUP BY COALESCE(onpromotion, false);
    """

logger.info("Loading on promotion items...")
promo_impact = pd.read_sql(query,engine)
logger.info("Loaded on promotion data")

print("\nPromotion Impact:")
promo_impact.head()

In [ ]:
promo_impact['on_promotion'] = promo_impact['on_promotion'].map({True: 'With Promotion', False: 'No Promotion'})

promo_avg = promo_impact[promo_impact['on_promotion'] == 'With Promotion']['avg_sales_per_transaction'].values[0]
no_promo_avg = promo_impact[promo_impact['on_promotion'] == 'No Promotion']['avg_sales_per_transaction'].values[0]
lift = ((promo_avg / no_promo_avg - 1) * 100)

print(f"\nPromotion Lift: {lift:+.1f}%")



fig = px.bar(promo_impact, x='on_promotion',y='avg_sales_per_transaction',text='avg_sales_per_transaction', title='Average Sales per Transaction: Promotion Impact')
fig.update_layout( xaxis_title='Promotion Status',yaxis_title='Avg Unit Sales per Transaction')
fig.show()


Store and Products Findings:

1. Store Performace:
    - The top performance stores are concentrated in Quito.
    - Store type 'A' shows highest average sales.
    - The top 5 stores(less than 10% of locations) generate the 25% of total sales.

2. Products:
    - The top 5 product families account for 78.7% of total sales.
    - 'Grocery I' is the best-selling category.
    - Non-Perishable products have higher sales volume than perishables products.

3. Promotion Impact:
    - Promotions drive a significant 63.7% lift in average transactions compared to non-promotional days.
